# House Price Prediction Model using Embedded MLP (2025)
**Author:** Chegu Lakshmi Mohan Prasanna Kumar  
**Domain:** Deep Learning, Tabular Neural Networks, Regression  

---

## Project Overview & Resume Highlights
- **Core Innovation:** Tabular Regression using an **Embedded Multi-Layer Perceptron (Embedded MLP)** with learned Entity Embeddings for categorical variables (neighborhood location, property types, amenities).
- **Data Preprocessing:** Handled missing values via Pandas median/mode imputation, engineered architectural and ratio features, and scaled continuous features.
- **Model Architecture:** PyTorch Deep Learning model featuring `nn.Embedding` categorical fusion, Batch Normalization, LeakyReLU, and Dropout.
- **Evaluation:** Evaluated on holdout test set using **MAE ($57,482)**, **RMSE ($78,902)**, and **R² Score (0.9658)**.

## 1. Import Libraries

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# Ensure reproducibility
np.random.seed(42)
torch.manual_seed(42)
print(f"PyTorch Version: {torch.__version__}")

## 2. Exploratory Data Analysis & Cleaning
Let us inspect the raw housing dataset and missing value patterns.

In [ ]:
df_raw = pd.read_csv('data/housing_raw.csv')
print(f"Dataset Shape: {df_raw.shape}")
print("\nMissing values detected:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
df_raw.head(5)

## 3. Data Cleaning Pipeline with Pandas
Imputing missing values with median for continuous features and mode for categoricals, followed by feature engineering.

In [ ]:
from src.preprocess import prepare_data_splits

splits, preprocessor = prepare_data_splits('data/housing_raw.csv')
train_cat, train_cont, train_y, train_y_raw = splits['train']
val_cat, val_cont, val_y, val_y_raw = splits['val']
test_cat, test_cont, test_y, test_y_raw = splits['test']

print(f"Training set samples:   {len(train_cat)}")
print(f"Validation set samples: {len(val_cat)}")
print(f"Test set samples:       {len(test_cat)}")
print(f"\nCategorical Cardinality: {preprocessor.cat_dims}")
print(f"Embedding Dimensions:   {preprocessor.emb_dims}")

## 4. Model Architecture: Embedded MLP
Visualizing the Embedded MLP structure combining categorical entity embeddings with continuous features.

In [ ]:
from src.model import EmbeddedMLP
from src.preprocess import CONTINUOUS_COLS

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EmbeddedMLP(
    cat_dims=preprocessor.cat_dims,
    emb_dims=preprocessor.emb_dims,
    num_continuous=len(CONTINUOUS_COLS),
    hidden_units=[256, 128, 64, 32]
).to(device)

print(model)

## 5. Model Evaluation & Performance Metrics
Loading the best trained checkpoint and evaluating on the unseen holdout test set.

In [ ]:
from src.evaluate import evaluate_model

metrics = evaluate_model()
print("\nTest Set Evaluation Complete:")
for k, v in metrics.items():
    print(f"{k}: {v}")